In [1]:
from SagaLibRegression import *

In [23]:
import os

def geotiff_to_sdat(geotiff_path, sdat_path):
    """
    Converts a GeoTIFF raster to SAGA .sdat format using GDAL.

    Parameters:
        geotiff_path (str): Path to the input GeoTIFF file (.tif).
        sdat_path (str): Path to the output .sdat file (without extension inference).
    """
    if not geotiff_path.endswith(".tif"):
        raise ValueError("Input file must be a .tif GeoTIFF file")

    # Ensure the output path has the .sdat extension
    if not sdat_path.endswith(".sdat"):
        raise ValueError("Output path must have a .sdat extension")

    if os.path.isfile(sdat_path):
        print(f'File already processed at {sdat_path}')
        return 

    # Convert using gdal_translate
    cmd = f'gdal_translate -of SAGA "{geotiff_path}" "{sdat_path}"'
    result = os.system(cmd)

    if result == 0:
        print(f'# Successfully converted "{geotiff_path}" to "{sdat_path}".')
    else:
        print(f'! Failed to convert "{geotiff_path}" to "{sdat_path}".')

def sdat_to_geotiff(sdat_path, geotiff_path, epsg_code=None):
    """
    epsg_code=4979
    Converts a SAGA .sdat file to a GeoTIFF using GDAL.

    Parameters:
        sdat_path (str): Path to the input .sdat file.
        geotiff_path (str): Path to the output GeoTIFF file.
        epsg_code (int or None): EPSG code for spatial reference. If None, keep original.
    """
    #if not sdat_path.endswith(".sdat"):
        #sdat_path = sdat_path.replace(".sgrd", ".sdat")

    if os.path.isfile(geotiff_path):
        print(f'! The file "{geotiff_path}" already exists.')
        return

    if epsg_code is not None:
        cmd = f'gdal_translate -a_srs EPSG:{epsg_code} -of GTiff "{sdat_path}" "{geotiff_path}"'
    else:
        # Do not assign spatial reference; keep whatever is present
        cmd = f'gdal_translate -of GTiff "{sdat_path}" "{geotiff_path}"'

    result = os.system(cmd)

    if result == 0:
        print(f'# Successfully converted "{sdat_path}" to "{geotiff_path}".')
    else:
        print(f'! Failed to convert "{sdat_path}" to "{geotiff_path}".')


In [29]:
tdem_fn = "/media/ljp238/12TBWolf/BRCHIEVE/ATROI/tif/N10E105/ID0/N10E105_tdem_dem_egm.tif"
edem_fn = "/media/ljp238/12TBWolf/BRCHIEVE/ATROI/tif/N10E105/ID0/N10E105_edem_egm.tif"
them_fn = "/media/ljp238/12TBWolf/BRCHIEVE/ATROI/tif/N10E105/ID0/N10E105_tdem_hem.tif"
tgap_fn = "/media/ljp238/12TBWolf/BRCHIEVE/ATROI/tif/N10E105/ID0/N10E105_tdem_dem_egm_v_3_gap.tif"
sen1_fn = "/media/ljp238/12TBWolf/BRCHIEVE/ATROI/tif/N10E105/ID0/N10E105_s1.tif"
sen2_fn = "/media/ljp238/12TBWolf/BRCHIEVE/ATROI/tif/N10E105/ID0/N10E105_s2.tif"
esar_fn = "/media/ljp238/12TBWolf/BRCHIEVE/ATROI/tif/N10E105/ID0/N10E105_esawc.tif"
esan_fn = "/media/ljp238/12TBWolf/BRCHIEVE/ATROI/tif/N10E105/ID0/N10E105_esawc_x.tif"
egmg_fn = "/media/ljp238/12TBWolf/BRCHIEVE/ATROI/tif/N10E105/ID0/N10E105_egm08.tif"
# focus on the functions of interest alone

In [ ]:
y = egmg_fn
x = edem_fn #add them_fn + esar_fn

In [31]:
from concurrent.futures import ProcessPoolExecutor
grid_list = [tdem_fn,edem_fn,them_fn,tgap_fn,esan_fn,esar_fn,egmg_fn]
for grid in grid_list:
    tif = grid#tdem_fn 
    sdat = tif.replace('.tif', '.sdat')
    geotiff_to_sdat(tif, sdat)

File already processed at /media/ljp238/12TBWolf/BRCHIEVE/ATROI/tif/N10E105/ID0/N10E105_tdem_dem_egm.sdat
File already processed at /media/ljp238/12TBWolf/BRCHIEVE/ATROI/tif/N10E105/ID0/N10E105_edem_egm.sdat
File already processed at /media/ljp238/12TBWolf/BRCHIEVE/ATROI/tif/N10E105/ID0/N10E105_tdem_hem.sdat
File already processed at /media/ljp238/12TBWolf/BRCHIEVE/ATROI/tif/N10E105/ID0/N10E105_tdem_dem_egm_v_3_gap.sdat
File already processed at /media/ljp238/12TBWolf/BRCHIEVE/ATROI/tif/N10E105/ID0/N10E105_esawc_x.sdat
File already processed at /media/ljp238/12TBWolf/BRCHIEVE/ATROI/tif/N10E105/ID0/N10E105_esawc.sdat
Input file size is 2259, 2056
0...10...20...30...40...50...60...70...80...90...100 - done.
# Successfully converted "/media/ljp238/12TBWolf/BRCHIEVE/ATROI/tif/N10E105/ID0/N10E105_egm08.tif" to "/media/ljp238/12TBWolf/BRCHIEVE/ATROI/tif/N10E105/ID0/N10E105_egm08.sdat".


In [28]:
import os
import time
import subprocess

def run_gwr_grid_downscaling(
    predictors,        # path to .sgrd list of predictor grids
    dependent,         # path to dependent grid (.sgrd)
    outdir,            # output directory where all results will be saved
    log_reg=0,         # use logistic regression: 0=no, 1=yes
    model_out=0,       # output model parameters: 0=no, 1=yes
    search_range=0,    # 0=local, 1=global
    search_radius=10,  # radius in cells for search
    dw_weighting=0,    # distance weighting method: 0-3
    dw_idw_power=2.0,  # IDW power
    dw_bandwidth=1.0,  # bandwidth for exponential/gaussian
    overwrite=False,   # if False, skip if output exists; if True, rerun
):
    """
    Runs SAGA-GIS Geographically Weighted Regression (GWR) for grid downscaling.

    Use case:
    Perform GWR using grid predictors and dependent variable grids.
    Outputs regression, residuals, coefficient of determination, and optionally model parameters.
    Supports logistic regression and various spatial weighting schemes.

    Returns path to output regression .sdat file.
    """
    os.makedirs(outdir, exist_ok=True)

    # Generate dynamic names based on input grid name and parameters
    base = os.path.splitext(os.path.basename(dependent))[0]
    suffix = f"gwr_l{log_reg}_r{search_radius}_w{dw_weighting}_b{dw_bandwidth}"
    
    regression = os.path.join(outdir, f"{base}_{suffix}_regression.sgrd")
    quality    = os.path.join(outdir, f"{base}_{suffix}_quality.sgrd")
    residuals  = os.path.join(outdir, f"{base}_{suffix}_residuals.sgrd")

    # Optional outputs
    reg_rescorr = os.path.join(outdir, f"{base}_{suffix}_rescorr.sgrd") if model_out else None
    model       = os.path.join(outdir, f"{base}_{suffix}_model.sgrd") if model_out else None

    # Check if output already exists
    regression_sdat = regression.replace('.sgrd', '.sdat')
    if os.path.exists(regression_sdat) and not overwrite:
        print(f"Skipped: {regression_sdat} exists.")
        return regression_sdat

    # Build command
    cmd = [
        "saga_cmd", "statistics_regression", "14",

        # Required inputs/outputs
        "-PREDICTORS", predictors,
        "-DEPENDENT", dependent,
        "-REGRESSION", regression,
        "-QUALITY", quality,
        "-RESIDUALS", residuals,

        # Optional parameters
        "-LOGISTIC", str(log_reg),
        "-MODEL_OUT", str(model_out),
        "-SEARCH_RANGE", str(search_range),
        "-SEARCH_RADIUS", str(search_radius),
        "-DW_WEIGHTING", str(dw_weighting),
        "-DW_IDW_POWER", str(dw_idw_power),
        "-DW_BANDWIDTH", str(dw_bandwidth),
    ]
    if reg_rescorr:
        cmd.extend(["-REG_RESCORR", reg_rescorr])
    if model:
        cmd.extend(["-MODEL", model])

    # Run the command
    start = time.time()
    subprocess.run(cmd, check=True)
    end = time.time()
    print(f"GWR Grid Downscaling completed in {end - start:.2f} seconds")

    return regression_sdat


In [62]:
import os
import time
import subprocess

def run_gwr_single_predictor_grid(
    points,                # input points shapefile
    dependent,             # dependent field name in shapefile
    predictor,             # predictor raster path (.sdat or .tif)
    outdir,                # output directory to store all outputs
    overwrite=False,       # whether to overwrite outputs if they already exist

    # Output file suffixes are auto-named based on the predictor filename
    log_reg=0,
    dw_weighting=0,
    dw_idw_power=2.0,
    dw_bandwidth=1.0,
    search_range=1,
    search_radius=1000.0,
    search_points_all=1,
    search_points_min=16,
    search_points_max=20,
    search_direction=0,
):
    """
    Run SAGA-GIS Geographically Weighted Regression (GWR) for a single predictor grid.
    
    All outputs are saved in `outdir`, which will be created if it doesn't exist.
    Output filenames are automatically named using the predictor's base name.
    If output already exists, the run is skipped unless `overwrite=True`.

    Returns the regression output `.sdat` path.
    """

    os.makedirs(outdir, exist_ok=True)

    # Get base name from predictor (e.g. predictor = '/path/to/elev.sdat' → 'elev')
    predictor_base = os.path.splitext(os.path.basename(predictor))[0]

    # Build output paths
    regression = os.path.join(outdir, f"regression_{predictor_base}.sdat")
    residuals  = os.path.join(outdir, f"residuals_{predictor_base}.shp")
    quality    = os.path.join(outdir, f"quality_{predictor_base}.sdat")
    intercept  = os.path.join(outdir, f"intercept_{predictor_base}.sdat")
    slope      = os.path.join(outdir, f"slope_{predictor_base}.sdat")

    # Skip execution if output exists unless overwrite=True
    if os.path.exists(regression) and not overwrite:
        print(f"[SKIPPED] GWR already exists for {predictor_base}")
        return regression

    cmd = [
        "saga_cmd", "statistics_regression", "4",

        # Required inputs/outputs
        "-POINTS", points,              # Input points shapefile; required
        "-DEPENDENT", dependent,        # Dependent variable field; required
        "-PREDICTOR", predictor,        # Predictor grid; required
        "-REGRESSION", regression,      # Output regression grid; required
        "-RESIDUALS", residuals,        # Residuals output shapes
        "-QUALITY", quality,            # Coefficient of determination grid
        "-INTERCEPT", intercept,        # Intercept grid
        "-SLOPE", slope,                # Slope grid

        # Options and flags
        "-LOGISTIC", str(log_reg),           # Logistic regression: 0=no, 1=yes
        "-DW_WEIGHTING", str(dw_weighting),  # Weighting function: 0=none, 1=IDW, 2=exp, 3=gauss
        "-DW_IDW_POWER", str(dw_idw_power),  # Power for IDW
        "-DW_BANDWIDTH", str(dw_bandwidth),  # Bandwidth for exp/gaussian
        "-SEARCH_RANGE", str(search_range),  # 0=local, 1=global
        "-SEARCH_RADIUS", str(search_radius),
        "-SEARCH_POINTS_ALL", str(search_points_all),
        "-SEARCH_POINTS_MIN", str(search_points_min),
        "-SEARCH_POINTS_MAX", str(search_points_max),
        "-SEARCH_DIRECTION", str(search_direction),
    ]

    start = time.time()
    subprocess.run(cmd, check=True)
    end = time.time()
    print(f"[DONE] GWR completed for {predictor_base} in {end - start:.2f} seconds")

    return regression


In [63]:
import os
import time
import subprocess

def close_gaps_saga(
    input_grid: str,            # Path to input .sgrd or .tif with NoData gaps
    outdir: str,                # Directory where output should be saved
    mask_grid: str = None,      # Optional mask to limit filling area
    threshold: float = 0.1,     # Surface tension threshold (lower is smoother)
    overwrite: bool = False     # Whether to overwrite existing outputs
) -> str:
    """
    Fills NoData (gap) cells in a grid using SAGA-GIS's 'Close Gaps' tool.
    """

    # Create output directory if it doesn't exist
    os.makedirs(outdir, exist_ok=True)

    # Derive base name for input grid
    base = os.path.splitext(os.path.basename(input_grid))[0]

    # Build a unique output name based on parameters
    mask_tag = f"_mask" if mask_grid else ""
    output_name = f"{base}_closegaps_thr{threshold}{mask_tag}.sdat"
    output_grid = os.path.join(outdir, output_name)

    # Check if output exists and skip unless overwrite=True
    if os.path.exists(output_grid) and not overwrite:
        print(f"Output exists, skipping: {output_grid}")
        return output_grid

    # Build SAGA-GIS command
    cmd = [
        "saga_cmd", "grid_tools", "7",
        "-INPUT", input_grid,
        "-THRESHOLD", str(threshold),
        "-RESULT", output_grid
    ]

    if mask_grid:
        cmd.extend(["-MASK", mask_grid])

    # Run the command
    start = time.time()
    subprocess.run(cmd, check=True)
    end = time.time()
    print(f"Gap filling completed in {end - start:.2f} seconds.")

    # Return path to .sdat file
    return output_grid


In [69]:
def fsaga(tif):
    saga = tif.replace('.tif', '.sgrd')
    #print('check that this file exist latter, and convert') both at sgrd and sdat- see what works best 
    return saga
    #geotiff_to_sdat(tif, sdat)

def fdir(tif, name):
    sdir = os.path.join(os.path.dirname(tif),name.upper())
    os.makedirs(sdir, exist_ok=True)
    return sdir

In [58]:
y = egmg_fn
x = tdem_fn #add them_fn + esar_fn
tdem_fn

'/media/ljp238/12TBWolf/BRCHIEVE/ATROI/tif/N10E105/ID0/N10E105_tdem_dem_egm.tif'

In [60]:
predictors = fsaga(tdem_fn)
dependent  = fsaga(egmg_fn)
outdir     = fdir(tdem_fn,'GWRd')
outdir

'/media/ljp238/12TBWolf/BRCHIEVE/ATROI/tif/N10E105/ID0/GWRd'

In [70]:
import os
import time
import subprocess
from pathlib import Path

def close_gaps_with_spline(
    input_grid: str,                        # required: path to input .sgrd file with gaps
    outdir: str,                            # required: output directory to save the results
    mask_grid: str = None,                  # optional: path to mask limiting fill area
    max_gap_cells: int = 0,                 # optional: max allowed gap size to fill (0 = all)
    max_points: int = 1000,                 # optional: max nearby known points for interpolation
    local_points: int = 20,                 # optional: local points used per interpolation
    extended_neighborhood: bool = False,    # optional: whether to use extended neighborhood
    neighbours: int = 0,                    # optional: 0 = Neumann, 1 = Moore
    radius: int = 0,                        # optional: neighborhood radius
    relaxation: float = 0.0,                # optional: interpolation smoothness
    overwrite: bool = False                 # optional: force re-run even if output exists
) -> str:
    """
    Fills NoData (gap) cells in a grid using spline-based local interpolation in SAGA-GIS.
    All outputs are saved to `outdir`. Returns the path to the .sdat file of the output.

    Parameters:
    ----------
    See inline comments for what each argument does.
    """
    
    # make sure output directory exists
    os.makedirs(outdir, exist_ok=True)

    # generate base output name based on parameters
    base_name = f"spline_closed_mgap{max_gap_cells}_mp{max_points}_lp{local_points}_ext{int(extended_neighborhood)}_nb{neighbours}_rad{radius}_rel{relaxation}"
    out_prefix = Path(input_grid).stem + "_" + base_name

    # construct output paths
    output_grid = os.path.join(outdir, out_prefix + ".sgrd")
    output_sdat = output_grid.replace(".sgrd", ".sdat")

    # check if output already exists
    if os.path.exists(output_sdat) and not overwrite:
        print(f"Output exists, skipping: {output_sdat}")
        return output_sdat

    cmd = [
        "saga_cmd", "grid_tools", "25",
        "-GRID", input_grid,
        "-MAXGAPCELLS", str(max_gap_cells),
        "-MAXPOINTS", str(max_points),
        "-LOCALPOINTS", str(local_points),
        "-EXTENDED", str(int(extended_neighborhood)),
        "-NEIGHBOURS", str(neighbours),
        "-RADIUS", str(radius),
        "-RELAXATION", str(relaxation),
        "-CLOSED", output_grid
    ]

    if mask_grid:
        cmd.extend(["-MASK", mask_grid])

    start = time.time()
    subprocess.run(cmd, check=True)
    end = time.time()
    print(f"Spline gap filling completed in {end - start:.2f} seconds.")
    
    return output_sdat


In [66]:
input_grid  = fsaga(tgap_fn)
outdir     = fdir(tgap_fn,'GAP')
outdir

'/media/ljp238/12TBWolf/BRCHIEVE/ATROI/tif/N10E105/ID0/GAP'

In [73]:
result = close_gaps_with_spline(
    input_grid= tgap_fn,#fsaga(tgap_fn),
    outdir= fdir(tgap_fn,'gaps_spline'),
    max_gap_cells=100,
    max_points=500,
    local_points=30,
    extended_neighborhood=True,
    neighbours=8,
    radius=8,
    relaxation=0.2,
    overwrite=False
)
print("Output written to:", result)


____________________________

   #####   ##   #####    ##
  ###     ###  ##       ###
   ###   # ## ##  #### # ##
    ### ##### ##    # #####
 ##### #   ##  ##### #   ##
____________________________

SAGA Version: 8.2.2

____________________________
library path: /usr/lib/x86_64-linux-gnu/saga/
library name: libgrid_tools
library     : grid_tools
tool        : Close Gaps with Spline
identifier  : 25
author      : O.Conrad (c) 2010
processors  : 56 [56]
____________________________

loading: N10E105_tdem_dem_egm_v_3_gap

100%
[Close Gaps with Spline] Execution started...

__________
[Close Gaps with Spline] Parameters:

Grid System: 0.000111; 2259x 2056y; 105.467333x 10.331778y
Grid: N10E105_tdem_dem_egm_v_3_gap
Mask: <not set>
Only Process Gaps with Less Cells: 100
Closed Gaps Grid: Closed Gaps Grid
Maximum Points: 500
Number of Points for Local Interpolation: 30
Extended Neighourhood: true
Neighbourhood: Neumann
Radius (Cells): 8
Relaxation: 0.2


100%

101%
__________
total execution

CalledProcessError: Command '['saga_cmd', 'grid_tools', '25', '-GRID', '/media/ljp238/12TBWolf/BRCHIEVE/ATROI/tif/N10E105/ID0/N10E105_tdem_dem_egm_v_3_gap.tif', '-MAXGAPCELLS', '100', '-MAXPOINTS', '500', '-LOCALPOINTS', '30', '-EXTENDED', '1', '-NEIGHBOURS', '8', '-RADIUS', '8', '-RELAXATION', '0.2', '-CLOSED', '/media/ljp238/12TBWolf/BRCHIEVE/ATROI/tif/N10E105/ID0/GAPS_SPLINE/N10E105_tdem_dem_egm_v_3_gap_spline_closed_mgap100_mp500_lp30_ext1_nb8_rad8_rel0.2.sgrd']' died with <Signals.SIGSEGV: 11>.

In [ ]:
sdat_path = close_gaps_saga(
    input_grid=input_grid,
    outdir=outdir,
    #mask_grid="valid_area.sgrd",
    threshold=0.05,
    overwrite=False
)
print(f"Output saved at: {sdat_path}")


In [61]:
run_gwr_grid_downscaling(
    predictors,        # path to .sgrd list of predictor grids
    dependent,         # path to dependent grid (.sgrd)
    outdir,            # output directory where all results will be saved
    log_reg=0,         # use logistic regression: 0=no, 1=yes
    model_out=1,       # output model parameters: 0=no, 1=yes
    search_range=0,    # 0=local, 1=global
    search_radius=10,  # radius in cells for search
    dw_weighting=0,    # distance weighting method: 0-3
    dw_idw_power=2.0,  # IDW power
    dw_bandwidth=1.0,  # bandwidth for exponential/gaussian
    overwrite=False,   # if False, skip if output exists; if True, rerun
)

____________________________

   #####   ##   #####    ##
  ###     ###  ##       ###
   ###   # ## ##  #### # ##
    ### ##### ##    # #####
 ##### #   ##  ##### #   ##
____________________________

SAGA Version: 8.2.2

____________________________
library path: /usr/lib/x86_64-linux-gnu/saga/
library name: libstatistics_regression
library     : statistics_regression
tool        : GWR for Grid Downscaling
identifier  : 14
author      : O.Conrad (c) 2013
processors  : 56 [56]
____________________________

Loading grid: /media/ljp238/12TBWolf/BRCHIEVE/ATROI/tif/N10E105/ID0/N10E105_tdem_dem_egm.sgrd...

100%
100%
okay
Loading grid: /media/ljp238/12TBWolf/BRCHIEVE/ATROI/tif/N10E105/ID0/N10E105_egm08.sgrd...

100%
100%
okay
[GWR for Grid Downscaling] Execution started...

__________
[GWR for Grid Downscaling] Parameters:

Grid System: 0.000111; 2259x 2056y; 105.467333x 10.331778y
Predictors: 1 object (N10E105_tdem_dem_egm)
Regression: Regression
Regression with Residual Correction: Regress

CalledProcessError: Command '['saga_cmd', 'statistics_regression', '14', '-PREDICTORS', '/media/ljp238/12TBWolf/BRCHIEVE/ATROI/tif/N10E105/ID0/N10E105_tdem_dem_egm.sgrd', '-DEPENDENT', '/media/ljp238/12TBWolf/BRCHIEVE/ATROI/tif/N10E105/ID0/N10E105_egm08.sgrd', '-REGRESSION', '/media/ljp238/12TBWolf/BRCHIEVE/ATROI/tif/N10E105/ID0/GWRd/N10E105_egm08_gwr_l0_r10_w0_b1.0_regression.sgrd', '-QUALITY', '/media/ljp238/12TBWolf/BRCHIEVE/ATROI/tif/N10E105/ID0/GWRd/N10E105_egm08_gwr_l0_r10_w0_b1.0_quality.sgrd', '-RESIDUALS', '/media/ljp238/12TBWolf/BRCHIEVE/ATROI/tif/N10E105/ID0/GWRd/N10E105_egm08_gwr_l0_r10_w0_b1.0_residuals.sgrd', '-LOGISTIC', '0', '-MODEL_OUT', '1', '-SEARCH_RANGE', '0', '-SEARCH_RADIUS', '10', '-DW_WEIGHTING', '0', '-DW_IDW_POWER', '2.0', '-DW_BANDWIDTH', '1.0', '-REG_RESCORR', '/media/ljp238/12TBWolf/BRCHIEVE/ATROI/tif/N10E105/ID0/GWRd/N10E105_egm08_gwr_l0_r10_w0_b1.0_rescorr.sgrd', '-MODEL', '/media/ljp238/12TBWolf/BRCHIEVE/ATROI/tif/N10E105/ID0/GWRd/N10E105_egm08_gwr_l0_r

In [ ]:
# run_cellwise_trend(
#     y_grids="grids/y_stack.sgrd",                     # Required input grid list
#     coeff="coefficients.sgrd",                        # Output coefficients grid
#     r2="r2.sgrd",                                      # Output R² grid
#     r2adj="r2adj.sgrd",                                # Output adjusted R² grid
#     stderr="stderr.sgrd",                              # Output standard error grid
#     p="pvalues.sgrd",                                  # Output p-values grid
#     linear=0,                                          # Use full polynomial trend, not just linear
#     order=3,                                           # Fit a 3rd order polynomial
#     xsource=1,                                         # Use table for independent variable
#     x_table="tables/years.txt",                        # Table with X values (e.g., years)
#     x_grids=None,                                      # Not needed since xsource != 2
#     outdir="outputs/cellwise_trend_run1"              # Save everything in this output folder
# )
